**Step 1:Installing Dependencies**

In [1]:
!pip install -q -U "transformers>=4.51.0" "huggingface_hub>=0.28.0" peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 45.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 32.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 58.8 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 36.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is 

**Step 2: Load Tokenizer & Dataset**

In [2]:
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 2. Load Dataset 1: MetaMathQA (20,000 rows)
ds1 = load_dataset("meta-math/MetaMathQA", split="train").shuffle(seed=42).select(range(20000))
ds1 = ds1.select_columns(["query", "response"])

# 3. Load Dataset 2: NuminaMath-CoT (15,000 rows) and map column names to match
ds2 = load_dataset("AI-MO/NuminaMath-CoT", split="train").shuffle(seed=42).select(range(15000))
ds2 = ds2.rename_columns({"problem": "query", "solution": "response"})
ds2 = ds2.select_columns(["query", "response"])

# 4. Concatenate and Shuffle Blended Dataset
ds = concatenate_datasets([ds1, ds2]).shuffle(seed=42)

print("--- BLENDED DATASET SUMMARY ---")
print(f"Total blended training rows: {len(ds)}")
print(f"Dataset columns: {ds.column_names}")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  166kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/859494 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

--- BLENDED DATASET SUMMARY ---
Total blended training rows: 35000
Dataset columns: ['query', 'response']


**Step 3: Checking Token Length Distribution (1024 Context Audit)**

In [3]:
# Analyzes 1,000 samples using the correct 'query' and 'response' columns for math

SYSTEM_PROMPT = (
    "You are an expert mathematician. Solve the problem step-by-step "
    "showing clear, logical reasoning, and state your final answer inside \\boxed{}."
)

def calculate_tokens(example):
    query_text = str(example["query"]).strip()
    response_text = str(example["response"]).strip()  # Mapped to 'response'

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query_text},
        {"role": "assistant", "content": response_text}
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        enable_thinking=False
    )
    
    tokens = tokenizer(formatted_text, truncation=False)["input_ids"]
    return {"token_count": len(tokens)}

# Sample 1,000 rows for fast auditing
sample_audit = ds.select(range(1000)).map(calculate_tokens)
lengths = sample_audit["token_count"]

valid_1024 = sum(1 for l in lengths if l <= 1024)
avg_len = sum(lengths) / len(lengths)

print("--- TOKEN LENGTH AUDIT (MATH BLEND) ---")
print(f"Average token length: {avg_len:.1f}")
print(f"Max token length in sample: {max(lengths)}")
print(f"Min token length in sample: {min(lengths)}")
print(f"Samples fitting within 1024 tokens: {valid_1024} / 1000 ({valid_1024/10:.1f}%)")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

--- TOKEN LENGTH AUDIT (MATH BLEND) ---
Average token length: 398.6
Max token length in sample: 2345
Min token length in sample: 97
Samples fitting within 1024 tokens: 973 / 1000 (97.3%)


**Step 4: Visual Inspection of 5 Random Formatted Examples**

In [4]:
import random

SYSTEM_PROMPT = (
    "You are an expert mathematician. Solve the problem step-by-step "
    "showing clear, logical reasoning, and state your final answer inside \\boxed{}."
)

print("--- AUDIT: VISUAL INSPECTION OF 5 RANDOM MATH EXAMPLES ---")

sample_indices = random.sample(range(len(ds)), 5)

for i, idx in enumerate(sample_indices, 1):
    sample = ds[idx]
    query_text = str(sample["query"]).strip()
    response_text = str(sample["response"]).strip()  # Mapped to 'query' and 'response'

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query_text},
        {"role": "assistant", "content": response_text}
    ]

    formatted = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        enable_thinking=False
    )

    print(f"\n==================== SAMPLE #{i} (Index {idx}) ====================")
    print(formatted[:600] + "\n...[TRUNCATED FOR DISPLAY]...")

--- AUDIT: VISUAL INSPECTION OF 5 RANDOM MATH EXAMPLES ---

==================== SAMPLE #1 (Index 5690) ====================
<|im_start|>system
You are an expert mathematician. Solve the problem step-by-step showing clear, logical reasoning, and state your final answer inside \boxed{}.<|im_end|>
<|im_start|>user
What is the largest prime factor of 999?<|im_end|>
<|im_start|>assistant
Since $999 = 3^3 \cdot 37$, the largest prime factor of 999 is $\boxed{37}$.
The answer is: 37<|im_end|>

...[TRUNCATED FOR DISPLAY]...

==================== SAMPLE #2 (Index 12795) ====================
<|im_start|>system
You are an expert mathematician. Solve the problem step-by-step showing clear, logical reasoning, and state your final answer inside \boxed{}.<|im_end|>
<|im_start|>user
Louise is hanging 30 of her pictures on the wall. She hangs some of them vertically, half of them horizontally, then hangs the remaining x pictures haphazardly. How many pictures did Louise hang vertically?
If we know the

**Step 5: Verify Response-Only Loss Masking (labels = -100)**

In [5]:
# Verifies that prompt tokens are masked with -100 using 'query' and 'response'

SYSTEM_PROMPT = (
    "You are an expert mathematician. Solve the problem step-by-step "
    "showing clear, logical reasoning, and state your final answer inside \\boxed{}."
)

sample = ds[0]
query_text = str(sample["query"]).strip()
response_text = str(sample["response"]).strip()  # Mapped to 'response'

prompt_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": query_text}
]
full_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": query_text},
    {"role": "assistant", "content": response_text}
]

prompt_str = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
full_str = tokenizer.apply_chat_template(full_messages, tokenize=False, enable_thinking=False)

prompt_ids = tokenizer(prompt_str, truncation=True, max_length=1024)["input_ids"]
full_ids = tokenizer(full_str, truncation=True, max_length=1024)["input_ids"]

prompt_len = len(prompt_ids)
labels = [-100] * prompt_len + full_ids[prompt_len:]

unmasked_token_ids = [t for t in labels if t != -100]
decoded_loss_target = tokenizer.decode(unmasked_token_ids, skip_special_tokens=False)

print("--- MASKING AUDIT VERIFICATION (MATH BLEND) ---")
print(f"Total Sequence Tokens: {len(full_ids)}")
print(f"Masked Prompt Tokens (-100): {prompt_len}")
print(f"Active Loss Training Tokens: {len(unmasked_token_ids)}")
print("\n--- DECODED LOSS TARGET (Should now show full math solution) ---")
print(decoded_loss_target[:300] + "\n...")

--- MASKING AUDIT VERIFICATION (MATH BLEND) ---
Total Sequence Tokens: 241
Masked Prompt Tokens (-100): 66
Active Loss Training Tokens: 175

--- DECODED LOSS TARGET (Should now show full math solution) ---
We can set up a proportion to solve this problem.
$\frac{2.5 \text{ cents}}{\text{page}} = \frac{\$20}{\text{unknown number of pages}}$
To convert $\$20$ to cents, we multiply by 100 to get $2000$ cents.
So, $\frac{2.5}{1} = \frac{2000}{\text{unknown number of pages}}$
Cross multiplying, we get $2.5
...


**Process Started**

**Step 1: Preprocessing & Filtering Cell**

In [6]:
import random

SYSTEM_PROMPT = (
    "You are an expert mathematician. Solve the problem step-by-step "
    "showing clear, logical reasoning, and state your final answer inside \\boxed{}."
)

# 1. Map and tokenize with hard filtering using 'query' and 'response'
def preprocess_math_data(example):
    query_text = str(example["query"]).strip()
    response_text = str(example["response"]).strip()

    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query_text}
    ]
    full_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query_text},
        {"role": "assistant", "content": response_text}
    ]

    prompt_str = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    full_str = tokenizer.apply_chat_template(
        full_messages, tokenize=False, enable_thinking=False
    )

    prompt_ids = tokenizer(prompt_str, truncation=False)["input_ids"]
    full_ids = tokenizer(full_str, truncation=False)["input_ids"]

    # Calculate token length
    total_len = len(full_ids)
    
    # Return empty if over context budget (filtered out in next step)
    if total_len > 768 or total_len < 30:
        return {"input_ids": [], "labels": [], "attention_mask": [], "valid": False}

    prompt_len = len(prompt_ids)
    labels = [-100] * prompt_len + full_ids[prompt_len:]

    return {
        "input_ids": full_ids,
        "labels": labels,
        "attention_mask": [1] * total_len,
        "valid": True
    }

print("Preprocessing and filtering dataset...")
processed_ds = ds.map(preprocess_math_data, remove_columns=ds.column_names)

# Filter out invalid/oversized rows
clean_ds = processed_ds.filter(lambda x: x["valid"]).remove_columns(["valid"])

# Shuffle and select 35,000 rows
train_ds = clean_ds.shuffle(seed=42).select(range(min(35000, len(clean_ds))))

print(f"Filtered Dataset Ready! Training samples: {len(train_ds)}")

Preprocessing and filtering dataset...


Map:   0%|          | 0/35000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/35000 [00:00<?, ? examples/s]

Filtered Dataset Ready! Training samples: 31921


**Step 2: Training Execution Cell (Creative LoRA)**

In [7]:
import os
import shutil
import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
OUTPUT_DIR = "./math_lora_qwen3"
FINAL_DIR = "./final_math_lora"

# 1. Quantization Config (NF4 4-bit with Double Quantization enabled)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 2. Load Base Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 3. Model Optimizations for QLoRA Training
model.gradient_checkpointing_enable()
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# 4. Apply LoRA Configuration (Higher capacity r=32, alpha=64 for Math reasoning)
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 5. Training Arguments (Cleaned for Kaggle Environment)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # Effective Batch Size = 16
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    gradient_checkpointing=False,    # Fast execution, zero quality loss
    lr_scheduler_type="cosine",
    warmup_steps=30,
    logging_steps=10,
    max_steps=1200,                  # ~4 hours runtime (~19,200 samples)
    fp16=True,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=2,
    seed=42,
    report_to="none"
)

# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    train_dataset=train_ds,
    args=training_args,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, 
        pad_to_multiple_of=8, 
        return_tensors="pt"
    )
)

# 7. Checkpoint Resume Logic & Execution
last_checkpoint = None
if os.path.exists(OUTPUT_DIR):
    checkpoints = [
        os.path.join(OUTPUT_DIR, d) 
        for d in os.listdir(OUTPUT_DIR) 
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        last_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]

if last_checkpoint:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting fresh Math LoRA Training...")
    trainer.train()

# 8. Save Weights and Package for Kaggle Persistence
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
shutil.make_archive("final_math_lora", 'zip', FINAL_DIR)

print(
    f"\nMath LoRA training complete!\n"
    f"1. Model saved locally to: {FINAL_DIR}\n"
    f"2. Packaged zip created: final_math_lora.zip (Downloadable from Kaggle Output directory)"
)

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157
Starting fresh Math LoRA Training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss
10,0.933350
20,0.375878
30,0.330193
40,0.322339
50,0.321446
60,0.311301
70,0.324296
80,0.313298
90,0.345124
100,0.256175


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt


Math LoRA training complete!
1. Model saved locally to: ./final_math_lora
2. Packaged zip created: final_math_lora.zip (Downloadable from Kaggle Output directory)


**Testing**

In [11]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 10.4 MB/s eta 0:00:0000:010:01


In [15]:
import gc
import time
import torch
from contextlib import nullcontext
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Flush VRAM cache
gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MATH_LORA_PATH = "./final_math_lora"

print("=== LOADING MODEL CLEANLY ON GPU ===")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Force entire model onto GPU 0 (prevents CPU offloading)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

model = PeftModel.from_pretrained(base_model, MATH_LORA_PATH)
model.eval()

benchmark_prompts = [
    {
        "id": "Q1",
        "category": "Algebra / Word Problem",
        "prompt": "A bakery sold 35% of its sourdough loaves in the morning and 40% of the remainder in the afternoon. If 78 loaves were left at the end of the day, how many loaves did the bakery start with?",
        "ground_truth": "200"
    },
    {
        "id": "Q2",
        "category": "Calculus (Integration by Parts)",
        "prompt": r"Evaluate the indefinite integral: \int x^2 \cdot e^{3x} \, dx. Provide the full solution with step-by-step substitution and constant of integration.",
        "ground_truth": r"\frac{1}{3}x^2 e^{3x} - \frac{2}{9}x e^{3x} + \frac{2}{27}e^{3x} + C"
    },
    {
        "id": "Q3",
        "category": "Number Theory (Modular Arithmetic)",
        "prompt": "Find the remainder when 7^2024 is divided by 25 using Euler's Totient Theorem.",
        "ground_truth": "1"
    },
    {
        "id": "Q4",
        "category": "Probability (Bayes' Theorem)",
        "prompt": "Disease X affects 1 in 1000 people. A test is 99% accurate for positive cases and 95% accurate for negative cases. If a random person tests positive, what is the exact probability they actually have Disease X?",
        "ground_truth": "1.94% (or ~0.0194)"
    },
    {
        "id": "Q5",
        "category": "Linear Algebra (Eigenvalues)",
        "prompt": "Find the eigenvalues of the matrix A = [[4, 1], [2, 3]]. Show the characteristic equation.",
        "ground_truth": "5 and 2"
    }
]

def generate_response(prompt: str, use_adapter: bool, max_tokens: int = 512):
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Dynamically match tensor placement to model device
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    context_mgr = nullcontext() if use_adapter else model.disable_adapter()
    
    t0 = time.time()
    with context_mgr:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=True,
                temperature=0.2,
                top_p=0.9
            )
    latency = time.time() - t0
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip(), latency

print("\n" + "=" * 80)
print("=== RUNNING BASE MODEL vs BASE MODEL + MATH ADAPTER BENCHMARK ===")
print("=" * 80)

for test in benchmark_prompts:
    print(f"\n[{test['id']} - {test['category']}]")
    print(f"Prompt: {test['prompt']}")
    print(f"Expected Answer: {test['ground_truth']}\n")
    
    # 1. Base Model Inference
    base_ans, base_time = generate_response(test["prompt"], use_adapter=False)
    
    # 2. Base + Math Adapter Inference
    adapter_ans, adapter_time = generate_response(test["prompt"], use_adapter=True)
    
    print("-" * 35 + " BASE MODEL ONLY " + "-" * 35)
    print(f"Latency: {base_time:.2f}s")
    print(base_ans)
    
    print("\n" + "-" * 33 + " BASE + MATH ADAPTER " + "-" * 33)
    print(f"Latency: {adapter_time:.2f}s")
    print(adapter_ans)
    print("=" * 80)

=== LOADING MODEL CLEANLY ON GPU ===


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


=== RUNNING BASE MODEL vs BASE MODEL + MATH ADAPTER BENCHMARK ===

[Q1 - Algebra / Word Problem]
Prompt: A bakery sold 35% of its sourdough loaves in the morning and 40% of the remainder in the afternoon. If 78 loaves were left at the end of the day, how many loaves did the bakery start with?
Expected Answer: 200

----------------------------------- BASE MODEL ONLY -----------------------------------
Latency: 18.98s
We are told:

- The bakery sold 35% of its sourdough loaves in the morning.
- Then sold 40% of the **remainder** in the afternoon.
- 78 loaves were left at the end of the day.

We are to find the original number of loaves.

---

### Step 1: Let the original number of loaves be $ x $.

**Morning sales:**  
35% sold in the morning → $ 0.35x $ sold  
So, remainder after morning = $ x - 0.35x = 0.65x $

**Afternoon sales:**  
40% of the remainder is sold → $ 0.40 \times 0.65x = 0.26x $ sold  
So, remainder after afternoon =  
$ 0.65x - 0.26x = 0.39x $

We are told that this fi